# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All references to fields, record sets, and columns use their respective `@id` fields to ensure accurate and reproducible data handling.

### Dataset Source
The dataset is defined by a Croissant schema, accessible from:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print basic dataset metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Let's review the available record sets and their fields. We use the `@id` of each entity for reference throughout this notebook.

> **Note**: Not all datasets populate the `recordSet` index in metadata. For the FAIR² dataset, let's enumerate accessible record sets directly from the dataset object.

In [ ]:
# Display all record sets and their @id's
print('Available record sets:')
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id}, name: {getattr(record_set, 'name', None)}")
    record_set_ids.append(record_set.id)

if not record_set_ids:
    print('No record sets found in metadata. Trying to enumerate available records directly...')

# For demonstration, try extracting one record set if available:
if record_set_ids:
    record_set_overview_id = record_set_ids[0]
    print(f'\nFields in record set {record_set_overview_id}:')
    for field in dataset.field_schema(record_set=record_set_overview_id):
        print(f"  - Field @id: {field['@id']}, name: {field['name']}")
else:
    print('No record sets to display fields from.')

## 3. Data Extraction
Load records for each available record set and convert them into pandas DataFrames using their `@id` fields. Each key in the resulting dictionary corresponds to a record set `@id`.

In [ ]:
# Extract records for all record sets, using @id references
dfs = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f"Columns (@id) in {record_set_id}:")
        print(list(df.columns))
    else:
        print(f"  No records found in {record_set_id}.")
# For illustration, select first record set with data
main_record_set_id = None
for rid, df in dfs.items():
    if not df.empty:
        main_record_set_id = rid
        break
if main_record_set_id:
    print(f"\nSample records from record set {main_record_set_id} (using @id reference):")
    display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Now we apply some basic EDA steps, referencing all fields by their `@id`. Example operations include filtering, normalizing, and grouping by attributes.

- Select a numeric field's `@id` and a group-field's `@id` from your DataFrame columns for demonstration.


In [ ]:
# Choose a numeric field and a group field by their @id
df = dfs[main_record_set_id]
print(f"Available columns (@id) in {main_record_set_id}:\n", list(df.columns))

# For illustration, try to infer a likely numeric field and grouping field
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Try to heuristic: numeric columns often have float/int types
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print('No numeric field detected for EDA.')
else:
    print(f"Using numeric field '@id': {numeric_field_id}")
    # Find a possible grouping field (categorical)
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == 'object' and df[col].nunique() < 10:
            group_field_id = col
            print(f"Using group field '@id': {group_field_id}")
            break

    # Filter, normalize, and group
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized values of {numeric_field_id}:\n")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the chosen group field (if found)
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (@id):")
        print(grouped_df.reset_index())

## 5. Visualization
Use matplotlib to visualize distributions of your chosen numeric field, and relationships if appropriate, referencing all fields by their `@id`.


In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=30, grid=False)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id is not None:
        plt.figure(figsize=(8,6))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded dataset metadata and explored available record sets, referencing all entities by `@id`.
- Extracted data tables for each record set and demonstrated how to access fields using their Croissant schema `@id`s.
- Performed exploratory data analysis through filtering, normalization, and grouping—again with all references by `@id`.
- Visualized distributions and group differences, showing the value of schema-driven, programmatic exploration.

You can apply similar steps to other FAIR Croissant datasets using their published schema URLs!